In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv("Social_Network_Ads.csv")

In [3]:
dataset.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [4]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [5]:
dataset=dataset.astype(int)

In [6]:
dataset.head()

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1


In [7]:
dataset=dataset.drop("User ID",axis=1)

In [8]:
dataset["Purchased"].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [9]:
indep=dataset[["Age","EstimatedSalary","Gender_Male"]]
dep=dataset["Purchased"]

In [10]:
indep.head()

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1


In [11]:
dep.head()

0    0
1    0
2    0
3    0
4    0
Name: Purchased, dtype: int32

In [12]:
#split into training set and test
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [13]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB

# Create an untuned KNN model
clf = GaussianNB()

# Define the parameter grid
param_grid = {
    'var_smoothing': np.logspace(0, -9, num=100)
}

# Create a GridSearchCV object
grid = GridSearchCV(clf, param_grid, cv=5)

# Fit the grid search object to the data
grid.fit(x_train, y_train)


GridSearchCV(cv=5, estimator=GaussianNB(),
             param_grid={'var_smoothing': array([1.00000000e+00, 8.11130831e-01, 6.57933225e-01, 5.33669923e-01,
       4.32876128e-01, 3.51119173e-01, 2.84803587e-01, 2.31012970e-01,
       1.87381742e-01, 1.51991108e-01, 1.23284674e-01, 1.00000000e-01,
       8.11130831e-02, 6.57933225e-02, 5.33669923e-02, 4.32876128e-02,
       3.51119173e-02, 2.84803587e-02, 2.31...
       1.23284674e-07, 1.00000000e-07, 8.11130831e-08, 6.57933225e-08,
       5.33669923e-08, 4.32876128e-08, 3.51119173e-08, 2.84803587e-08,
       2.31012970e-08, 1.87381742e-08, 1.51991108e-08, 1.23284674e-08,
       1.00000000e-08, 8.11130831e-09, 6.57933225e-09, 5.33669923e-09,
       4.32876128e-09, 3.51119173e-09, 2.84803587e-09, 2.31012970e-09,
       1.87381742e-09, 1.51991108e-09, 1.23284674e-09, 1.00000000e-09])})

In [16]:
# Get the best estimator
best_params = grid.best_params_

In [17]:
# Print the best parameters

print(best_params)

{'var_smoothing': 0.03511191734215131}


In [18]:
re=grid.cv_results_

In [20]:
grid_predictions=grid.predict(x_test)

In [22]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,grid_predictions)
        

In [24]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [25]:
print(clf_report)

              precision    recall  f1-score   support

           0       0.90      0.95      0.93        85
           1       0.91      0.82      0.86        49

    accuracy                           0.90       134
   macro avg       0.90      0.88      0.89       134
weighted avg       0.90      0.90      0.90       134



In [26]:
print(cm)

[[81  4]
 [ 9 40]]


In [27]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'var_smoothing': 0.03511191734215131}: 0.9017630740307678


In [28]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])


0.9603841536614646

In [29]:
table=pd.DataFrame.from_dict(re)

In [30]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_var_smoothing,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000964,0.000893,0.001370,0.001697,1.0,{'var_smoothing': 1.0},0.777778,0.773585,0.679245,0.886792,0.811321,0.785744,0.066960,100
1,0.002113,0.001967,0.000622,0.000830,0.811131,{'var_smoothing': 0.8111308307896871},0.777778,0.792453,0.679245,0.886792,0.849057,0.797065,0.070752,99
2,0.003145,0.006290,0.000198,0.000395,0.657933,{'var_smoothing': 0.657933224657568},0.777778,0.792453,0.698113,0.905660,0.867925,0.808386,0.072606,96
3,0.003472,0.006945,0.003348,0.006189,0.53367,{'var_smoothing': 0.533669923120631},0.777778,0.773585,0.716981,0.905660,0.867925,0.808386,0.068571,96
4,0.003125,0.006249,0.000000,0.000000,0.432876,{'var_smoothing': 0.43287612810830584},0.777778,0.773585,0.698113,0.924528,0.867925,0.808386,0.079173,96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.003125,0.006250,0.000000,0.000000,0.0,{'var_smoothing': 2.310129700083158e-09},0.833333,0.867925,0.792453,0.962264,0.981132,0.887421,0.073086,1
96,0.000000,0.000000,0.003124,0.006248,0.0,{'var_smoothing': 1.873817422860387e-09},0.833333,0.867925,0.792453,0.962264,0.981132,0.887421,0.073086,1
97,0.000000,0.000000,0.003124,0.006249,0.0,{'var_smoothing': 1.519911082952933e-09},0.833333,0.867925,0.792453,0.962264,0.981132,0.887421,0.073086,1
98,0.006248,0.007653,0.000000,0.000000,0.0,{'var_smoothing': 1.2328467394420635e-09},0.833333,0.867925,0.792453,0.962264,0.981132,0.887421,0.073086,1


In [31]:
#Future prediction

In [32]:
age=float(input("Age:"))
Estimated_salary=float(input("Estimated_salary:"))
sex_male=int(input("Sex Male 0 or 1:"))


Age:55
Estimated_salary:600000
Sex Male 0 or 1:1


In [33]:
Future_Prediction=grid.predict([[age,Estimated_salary,sex_male]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[1]


In [37]:
import pickle

In [38]:
filename= "finalized_model_Grid_GaussianNB_classification.sav"

In [39]:
pickle.dump(grid,open(filename,'wb'))

In [40]:
loaded_model = pickle.load(open("finalized_model_Grid_GaussianNB_classification.sav",'rb'))

In [41]:
result= loaded_model.predict([[age,Estimated_salary,sex_male]])

In [42]:
result

array([1])